In [86]:
import pandas as pd
import numpy as np
import matplotlib as plt

data_main = pd.read_csv('sources/upload-weeklyshipcrossingsforsixmaritimepassagesofinterest.csv')
data_excel = pd.read_excel(
    'sources/uktradeflowsofcontainerisedproductsthroughglobalmaritimepassages20202024.xlsx',
    sheet_name=['1.Monthly Volumes All (Imports)', '2.Monthly Volumes All (Exports)']
)
data_second = pd.concat(data_excel.values(), ignore_index=True)

In [87]:
data_main.head()

,Passage,Week of entry,Number of crossings
0,Bab-Al Mandab Strait,03/01/2022,394
1,Bab-Al Mandab Strait,10/01/2022,391
2,Bab-Al Mandab Strait,17/01/2022,386
3,Bab-Al Mandab Strait,24/01/2022,394
4,Bab-Al Mandab Strait,31/01/2022,415


In [88]:
data_main.info()

<class 'pandas.DataFrame'>
RangeIndex: 695 entries, 0 to 694
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Passage              695 non-null    str  
 1   Week of entry        695 non-null    str  
 2   Number of crossings  695 non-null    int64
dtypes: int64(1), str(2)
memory usage: 16.4 KB


In [89]:
data_second.head()

,Passage,Direction,TEU,Year,Month
0,Taiwan Strait,import,13625.1,2020,1
1,Taiwan Strait,import,2710.4,2020,2
2,Taiwan Strait,import,11166.5,2020,3
3,Taiwan Strait,import,7766,2020,4
4,Taiwan Strait,import,11118.4,2020,5


In [90]:
data_second.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Passage    600 non-null    str   
 1   Direction  600 non-null    str   
 2   TEU        600 non-null    object
 3   Year       600 non-null    int64 
 4   Month      600 non-null    int64 
dtypes: int64(2), object(1), str(2)
memory usage: 23.6+ KB


In [91]:
# Convert "Week of entry" to datetime (day/month/year format)
data_main['Week of entry'] = pd.to_datetime(data_main['Week of entry'], format='%d/%m/%Y')

data_main.head(10)


,Passage,Week of entry,Number of crossings
0,Bab-Al Mandab Strait,2022-01-03,394
1,Bab-Al Mandab Strait,2022-01-10,391
2,Bab-Al Mandab Strait,2022-01-17,386
3,Bab-Al Mandab Strait,2022-01-24,394
4,Bab-Al Mandab Strait,2022-01-31,415
5,Bab-Al Mandab Strait,2022-02-07,443
6,Bab-Al Mandab Strait,2022-02-14,364
7,Bab-Al Mandab Strait,2022-02-21,424
8,Bab-Al Mandab Strait,2022-02-28,394
9,Bab-Al Mandab Strait,2022-03-07,418


In [92]:
data_main.info()

<class 'pandas.DataFrame'>
RangeIndex: 695 entries, 0 to 694
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Passage              695 non-null    str           
 1   Week of entry        695 non-null    datetime64[us]
 2   Number of crossings  695 non-null    int64         
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 16.4 KB


In [93]:
# 1. Replace "x" in TEU with NaN, then convert to numeric
data_second['TEU'] = pd.to_numeric(data_second['TEU'].replace('x', np.nan))

# 2. Combine Year and Month into a datetime column
data_second['Date'] = pd.to_datetime(data_second[['Year', 'Month']].assign(day=1))

# 3. Group by Passage and Month, summing TEU
data_second = data_second.groupby(
    ['Passage', data_second['Date'].dt.to_period('M')]
).agg({'TEU': 'sum'}).reset_index()

data_second.head(10)

,Passage,Date,TEU
0,Cape of Good Hope,2020-01,511.2
1,Cape of Good Hope,2020-02,334.7
2,Cape of Good Hope,2020-03,730.3
3,Cape of Good Hope,2020-04,1493.4
4,Cape of Good Hope,2020-05,4806.4
5,Cape of Good Hope,2020-06,2669.0
6,Cape of Good Hope,2020-07,5794.9
7,Cape of Good Hope,2020-08,4294.9
8,Cape of Good Hope,2020-09,1857.6
9,Cape of Good Hope,2020-10,1045.2


In [94]:
data_main.to_csv('sources/data_main.csv', index=False)
data_second.to_csv('sources/data_second.csv', index=False)